# v8 - Unfreeze dos 2 ultimos blocos do SigLIP (AMP)

Experimento: descongelar os **2 ultimos blocos transformer** do encoder MedSigLIP e
treina-los com LR baixo (1e-5), mantendo o resto congelado. Comparar contra o baseline
congelado (v7, macro-F1 derm_test = 0,61).

**Pacote de mudancas (v8 muda varios fatores de uma vez, nao eh ablacao limpa):**
- Unfreeze dos 2 ultimos blocos + LR discriminativo (cabeca 5e-4 / encoder 1e-5)
- Augmentation REATIVADA no train (cache saiu)
- `WeightedRandomSampler` para balancear os batches (mantem todos os dados + class weights)
- Corte de NEV em 1400 mantido (mesma distribuicao do v7)

**Estabilidade numerica (AMP):** o modelo eh carregado em fp16; treinar fp16 puro pelo
encoder gera NaN (overflow na atencao). E fp32 puro eh ~8x mais lento no T4 (sem tensor
cores). Solucao: master weights em fp32 (`model.float()`) + `autocast` (compute fp16
rapido) + `GradScaler` (escala a loss, evita underflow). Sem cache de embeddings.

In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"  # reduz fragmentacao
import sys, site, importlib, subprocess
REPO_URL = "https://github.com/RodrigoAraujo12/melanoma-tcc.git"
REPO_DIR = "/kaggle/working/melanoma-tcc"
if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", REPO_DIR], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U",
                "transformers", "accelerate", "huggingface_hub"], check=True)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
importlib.invalidate_caches(); site.main()
print("Setup OK")

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, WeightedRandomSampler
from torch.optim import AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.cuda.amp import autocast, GradScaler
from collections import Counter
from sklearn.metrics import f1_score, recall_score, classification_report
from kaggle_secrets import UserSecretsClient

from melanoma_tcc.data.preprocessing import (
    Derm7ptUnifiedDataset, HAM10000Dataset, CombinedDermDataset,
    classification_collate_fn,
    GROUP_TO_LABEL, LABEL_TO_GROUP, METADATA_DIM_V5,
    HAM_DX_TO_GROUP, ham10000_train_val_split,
)
from melanoma_tcc.model.classifier import build_dermclassifier
from melanoma_tcc.model.losses import FocalLoss, compute_class_weights
from melanoma_tcc.utils.metrics import plot_confusion_matrix

secrets = UserSecretsClient()
HF_TOKEN = secrets.get_secret('mellanoma_TCC')

DERM7PT_DIR = "/kaggle/input/datasets/rodrigoadesouza/derm7pt-multimodal/release_v0"
DERM_META = f"{DERM7PT_DIR}/meta/meta.csv"
DERM_IMAGES = f"{DERM7PT_DIR}/images"
DERM_TRAIN_IDX = f"{DERM7PT_DIR}/meta/train_indexes.csv"
DERM_VAL_IDX = f"{DERM7PT_DIR}/meta/valid_indexes.csv"
DERM_TEST_IDX = f"{DERM7PT_DIR}/meta/test_indexes.csv"
HAM_DIR = "/kaggle/input/datasets/kmader/skin-cancer-mnist-ham10000"
HAM_META = f"{HAM_DIR}/HAM10000_metadata.csv"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MEL_LABEL = GROUP_TO_LABEL['MEL']
TARGET_NAMES = ["BCC", "NEV", "MEL", "SK", "MISC"]
print(f"Device: {device} | MEL label = {MEL_LABEL} | metadata_dim = {METADATA_DIM_V5}")

In [ ]:
# ===== Build model + fp32 master weights + DESCONGELA os 2 ultimos blocos =====
model, processor = build_dermclassifier(hf_token=HF_TOKEN, num_classes=5,
                                        metadata_dim=METADATA_DIM_V5, freeze_vision=True)
model = model.to(device).float()   # fp32 master weights (AMP faz o compute em fp16)
model.vision_encoder = model.vision_encoder.to(device)
print("dtype do encoder:", next(model.vision_encoder.parameters()).dtype)
print(f"GPU alocada apos build: {torch.cuda.memory_allocated()/1e9:.2f} GB")

print("\nType do vision_encoder:", type(model.vision_encoder).__name__)
print("ModuleLists encontradas no encoder:")
for name, mod in model.vision_encoder.named_modules():
    if isinstance(mod, nn.ModuleList) and len(mod) > 0:
        print(f"  {name}: {len(mod)} x {type(mod[0]).__name__}")

def find_encoder_layers(venc):
    getters = [
        ('vision_model.encoder.layers', lambda m: m.vision_model.encoder.layers),
        ('encoder.layers', lambda m: m.encoder.layers),
        ('layers', lambda m: m.layers),
    ]
    for path, g in getters:
        try:
            layers = g(venc)
            if isinstance(layers, nn.ModuleList) and len(layers) > 0:
                print(f"\n--> Usando vision_encoder.{path} ({len(layers)} blocos)")
                return layers
        except AttributeError:
            continue
    best, best_name = None, None
    for name, mod in venc.named_modules():
        if isinstance(mod, nn.ModuleList) and len(mod) > 0:
            if best is None or len(mod) > len(best):
                best, best_name = mod, name
    if best is None:
        raise RuntimeError("Nenhuma ModuleList encontrada no vision_encoder")
    print(f"\n--> [fallback] maior ModuleList: vision_encoder.{best_name} ({len(best)} blocos)")
    return best

layers = find_encoder_layers(model.vision_encoder)

for p in model.vision_encoder.parameters():
    p.requires_grad = False
unfrozen_blocks = [layers[-2], layers[-1]]
for blk in unfrozen_blocks:
    for p in blk.parameters():
        p.requires_grad = True

model._freeze_vision = False   # deixa o grad fluir pelo encode_image

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
enc_trainable = sum(p.numel() for blk in unfrozen_blocks for p in blk.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"\nParams treinaveis: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")
print(f"  dos quais nos 2 blocos do encoder: {enc_trainable:,}")

In [ ]:
# ===== Datasets: corte NEV=1400 (igual v7) + augment REATIVADA no train =====
NEV_TARGET = 1400

derm_train = Derm7ptUnifiedDataset(DERM_META, DERM_IMAGES, processor,
                                   indexes_csv=DERM_TRAIN_IDX, augment=True, seed=42)
derm_val = Derm7ptUnifiedDataset(DERM_META, DERM_IMAGES, processor,
                                 indexes_csv=DERM_VAL_IDX, augment=False)
derm_test = Derm7ptUnifiedDataset(DERM_META, DERM_IMAGES, processor,
                                  indexes_csv=DERM_TEST_IDX, augment=False)

ham_train_df, ham_val_df = ham10000_train_val_split(HAM_META, val_ratio=0.15, seed=42,
                                                    filter_unknown=True)
ham_train_df = ham_train_df.copy()
ham_train_df['group'] = (ham_train_df['dx'].str.strip().str.lower()
                         .map(HAM_DX_TO_GROUP).fillna('MISC'))
nev_mask = ham_train_df['group'] == 'NEV'
n_nev = int(nev_mask.sum())
if n_nev > NEV_TARGET:
    keep_nev = ham_train_df[nev_mask].sample(n=NEV_TARGET, random_state=42)
    ham_train_df = pd.concat([keep_nev, ham_train_df[~nev_mask]]).reset_index(drop=True)

ham_train = HAM10000Dataset(ham_train_df, HAM_DIR, processor, augment=True, seed=42)
ham_val = HAM10000Dataset(ham_val_df, HAM_DIR, processor, augment=False)

train_dataset = CombinedDermDataset([derm_train, ham_train])

train_groups = pd.concat([derm_train.df['group'], ham_train.df['group']])
print("Distribuicao TRAIN combinado:", dict(Counter(train_groups)))
print(f"Train: {len(train_dataset)} = derm({len(derm_train)}) + ham({len(ham_train)})")
print(f"Derm val: {len(derm_val)} | HAM val: {len(ham_val)} | Derm test: {len(derm_test)}")

# WeightedRandomSampler: labels na ORDEM do CombinedDermDataset
train_group_list = list(derm_train.df['group']) + list(ham_train.df['group'])
train_label_list = [GROUP_TO_LABEL[g] for g in train_group_list]
assert len(train_label_list) == len(train_dataset), "desalinhamento de indices!"
label_count = Counter(train_label_list)
sample_weights = [1.0 / label_count[lbl] for lbl in train_label_list]
sampler = WeightedRandomSampler(
    weights=torch.as_tensor(sample_weights, dtype=torch.double),
    num_samples=len(train_dataset), replacement=True,
)
print("\nPesos por classe (1/contagem):",
      {LABEL_TO_GROUP[l]: round(1.0/label_count[l], 5) for l in sorted(label_count)})

In [ ]:
BATCH_SIZE = 16   # AMP cabe em 16; se der OOM, baixe para 8
EPOCHS = 15
PATIENCE = 4
FOCAL_GAMMA = 2.0
LABEL_SMOOTHING = 0.05
LR_HEAD = 5e-4
LR_ENCODER = 1e-5
CONSERVATIVE_TAU = 0.40

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=sampler,
                          collate_fn=classification_collate_fn, num_workers=4, pin_memory=True)
derm_val_loader = DataLoader(derm_val, batch_size=BATCH_SIZE, shuffle=False,
                             collate_fn=classification_collate_fn, num_workers=4, pin_memory=True)
derm_test_loader = DataLoader(derm_test, batch_size=BATCH_SIZE, shuffle=False,
                              collate_fn=classification_collate_fn, num_workers=4, pin_memory=True)
ham_val_loader = DataLoader(ham_val, batch_size=BATCH_SIZE, shuffle=False,
                            collate_fn=classification_collate_fn, num_workers=4, pin_memory=True)

class_counts = [Counter(train_groups).get(LABEL_TO_GROUP[i], 1) for i in range(5)]
alpha = compute_class_weights(class_counts, mode="inverse_sqrt")
print(f"Class counts: {[(LABEL_TO_GROUP[i], class_counts[i]) for i in range(5)]}")
print(f"Class weights (alpha, inverse_sqrt): {alpha.tolist()}")
criterion = FocalLoss(alpha=alpha, gamma=FOCAL_GAMMA, label_smoothing=LABEL_SMOOTHING)

head_params = [p for m in (model.vision_proj, model.metadata_encoder, model.classifier)
               for p in m.parameters() if p.requires_grad]
encoder_params = [p for blk in unfrozen_blocks for p in blk.parameters() if p.requires_grad]
trainable_params = head_params + encoder_params
optimizer = AdamW([
    {'params': head_params, 'lr': LR_HEAD},
    {'params': encoder_params, 'lr': LR_ENCODER},
], weight_decay=0.01)
scheduler = ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=2)
scaler = GradScaler()   # AMP: escala a loss para evitar underflow do fp16
print(f"\nParam groups -> cabeca: {sum(p.numel() for p in head_params):,} @ {LR_HEAD} | "
      f"encoder: {sum(p.numel() for p in encoder_params):,} @ {LR_ENCODER}")

In [ ]:
# ===== Treino com AMP (autocast + GradScaler) =====
def set_train_modes(model):
    model.train()
    model.vision_encoder.eval()
    for blk in unfrozen_blocks:
        blk.train()

def evaluate(model, loader, criterion):
    model.eval()
    saved = model._freeze_vision
    model._freeze_vision = True
    total_loss, preds, labels = 0.0, [], []
    with torch.no_grad(), autocast():
        for batch in loader:
            pv = batch['pixel_values'].to(device)
            md = batch['metadata'].to(device)
            lb = batch['labels'].to(device)
            logits = model(pv, md)
            loss = criterion(logits, lb)
            total_loss += loss.item() * lb.size(0)
            preds.extend(logits.argmax(dim=-1).cpu().tolist())
            labels.extend(lb.cpu().tolist())
    model._freeze_vision = saved
    n = len(loader.dataset)
    acc = sum(int(p == l) for p, l in zip(preds, labels)) / n
    macro_f1 = f1_score(labels, preds, average='macro')
    return total_loss / n, acc, macro_f1, preds, labels

best_val_f1, best_state, best_epoch, no_improve = 0.0, None, 0, 0
trainable_keys = {n for n, p in model.named_parameters() if p.requires_grad}
history = []

for epoch in range(1, EPOCHS + 1):
    set_train_modes(model)
    tl_sum, tc, tt = 0.0, 0, 0
    for bi, batch in enumerate(train_loader):
        pv = batch['pixel_values'].to(device)
        md = batch['metadata'].to(device)
        lb = batch['labels'].to(device)
        optimizer.zero_grad()
        with autocast():
            logits = model(pv, md)
            loss = criterion(logits, lb)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)                                   # antes do clip
        torch.nn.utils.clip_grad_norm_(trainable_params, 1.0)
        scaler.step(optimizer)
        scaler.update()
        tl_sum += loss.item() * lb.size(0)
        tc += (logits.argmax(dim=-1) == lb).sum().item()
        tt += lb.size(0)
        if bi % 50 == 0:
            print(f"  ep{epoch} batch {bi}/{len(train_loader)} loss={loss.item():.4f}")
    train_loss, train_acc = tl_sum / tt, tc / tt

    val_loss, val_acc, val_f1, _, _ = evaluate(model, derm_val_loader, criterion)
    scheduler.step(val_f1)
    lr_head = optimizer.param_groups[0]['lr']
    lr_enc = optimizer.param_groups[1]['lr']
    history.append((epoch, train_loss, train_acc, val_loss, val_acc, val_f1, lr_head, lr_enc))
    print(f"Epoch {epoch:2d}/{EPOCHS} | train_loss={train_loss:.4f} acc={train_acc:.3f} | "
          f"val_loss={val_loss:.4f} acc={val_acc:.3f} macroF1={val_f1:.4f} | "
          f"lr_h={lr_head:.1e} lr_e={lr_enc:.1e}")

    if val_f1 > best_val_f1:
        best_val_f1, best_epoch, no_improve = val_f1, epoch, 0
        best_state = {k: v.detach().cpu().clone()
                      for k, v in model.state_dict().items() if k in trainable_keys}
    else:
        no_improve += 1
        if no_improve >= PATIENCE:
            print(f"\nEarly stopping na epoch {epoch} (sem melhora ha {PATIENCE} epocas).")
            break

print(f"\nBest macro-F1 (derm_val): {best_val_f1:.4f} na epoch {best_epoch}")

In [ ]:
if best_state is not None:
    current = model.state_dict()
    current.update(best_state)
    model.load_state_dict(current, strict=False)
    print(f"Best model carregado ({len(best_state)} tensores: cabeca + 2 blocos).")

os.makedirs("/kaggle/working/derm-classifier-v8", exist_ok=True)
torch.save(best_state, "/kaggle/working/derm-classifier-v8/trainable_state.pt")
print("Salvou /kaggle/working/derm-classifier-v8/trainable_state.pt")
import json
with open("/kaggle/working/derm-classifier-v8/history.json", "w") as f:
    json.dump(history, f, indent=2)

In [ ]:
# ===== Sweep de threshold do MEL no DERM_VAL =====
@torch.no_grad()
def predict_mel_threshold(model, loader, tau):
    model.eval()
    saved = model._freeze_vision
    model._freeze_vision = True
    Y, P = [], []
    with autocast():
        for batch in loader:
            pv = batch['pixel_values'].to(device)
            md = batch['metadata'].to(device)
            probs = F.softmax(model(pv, md).float(), dim=-1)
            am = probs.argmax(dim=-1)
            pred = torch.where(probs[:, MEL_LABEL] >= tau, torch.full_like(am, MEL_LABEL), am)
            Y.extend(batch['labels'].tolist())
            P.extend(pred.cpu().tolist())
    model._freeze_vision = saved
    return Y, P

def mel_prec(y, p):
    den = max(1, sum(1 for b in p if b == MEL_LABEL))
    return sum(1 for a, b in zip(y, p) if b == MEL_LABEL and a == MEL_LABEL) / den

print("Sweep de tau no DERM_VAL (tau>1 = argmax):")
print(f"{'tau':>6} | {'MEL_recall':>10} | {'MEL_prec':>9} | {'macroF1':>8} | {'acc':>6}")
for tau in [1.01, 0.50, 0.45, 0.40, 0.35, 0.30, 0.25, 0.20]:
    y, p = predict_mel_threshold(model, derm_val_loader, tau)
    rec = recall_score(y, p, labels=[MEL_LABEL], average='macro', zero_division=0)
    macro = f1_score(y, p, average='macro')
    acc = sum(int(a == b) for a, b in zip(y, p)) / len(y)
    tag = ' (argmax)' if tau > 1 else ''
    print(f"{tau:>6.2f} | {rec:>10.3f} | {mel_prec(y,p):>9.3f} | {macro:>8.4f} | {acc:>6.3f}{tag}")

In [ ]:
# ===== Avaliacao final nos dois dominios (argmax) =====
print("=" * 64)
print("DOMINIO 1: Derm7pt TEST (comparar com baseline v7 = macro-F1 0,612)")
print("=" * 64)
_, d_acc, d_f1, d_preds, d_labels = evaluate(model, derm_test_loader, criterion)
print(f"accuracy = {d_acc:.4f} | macro-F1 = {d_f1:.4f}\n")
print(classification_report(d_labels, d_preds, target_names=TARGET_NAMES, digits=4, zero_division=0))
plot_confusion_matrix(d_labels, d_preds, target_names=TARGET_NAMES,
                      save_path='/kaggle/working/derm-classifier-v8-derm-cm.png')

print("\n" + "=" * 64)
print("DOMINIO 2: HAM TEST (ham_val)")
print("=" * 64)
_, h_acc, h_f1, h_preds, h_labels = evaluate(model, ham_val_loader, criterion)
print(f"accuracy = {h_acc:.4f} | macro-F1 = {h_f1:.4f}\n")
print(classification_report(h_labels, h_preds, target_names=TARGET_NAMES, digits=4, zero_division=0))
plot_confusion_matrix(h_labels, h_preds, target_names=TARGET_NAMES,
                      save_path='/kaggle/working/derm-classifier-v8-ham-cm.png')

pd.DataFrame({'true_label': d_labels, 'pred_label': d_preds,
              'true_group': [LABEL_TO_GROUP[l] for l in d_labels],
              'pred_group': [LABEL_TO_GROUP[p] for p in d_preds]}
             ).to_csv('/kaggle/working/derm_classifier_v8_derm_predictions.csv', index=False)
pd.DataFrame({'true_label': h_labels, 'pred_label': h_preds,
              'true_group': [LABEL_TO_GROUP[l] for l in h_labels],
              'pred_group': [LABEL_TO_GROUP[p] for p in h_preds]}
             ).to_csv('/kaggle/working/derm_classifier_v8_ham_predictions.csv', index=False)
print("\nPredicoes salvas (derm + ham).")